In [ ]:
!pip -q install --force-reinstall --no-deps 'torch==2.4.1' 'torchvision==0.19.1' 'torchaudio==2.4.1' --index-url https://download.pytorch.org/whl/cu121
!pip -q install --upgrade 'transformers==4.51.3' accelerate 'bitsandbytes==0.43.3'

In [ ]:
import os, subprocess, sys
REPO = '/kaggle/working/lawforge'
if not os.path.isdir(REPO):
    subprocess.check_call(['git','clone','--depth','1','https://github.com/PAMF2/lawforge.git', REPO])
sys.path.insert(0, REPO)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL = 'Qwen/Qwen2.5-7B-Instruct'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
    device_map='cuda', trust_remote_code=True)
model.eval()
print(f'loaded {MODEL} mem={torch.cuda.memory_allocated()/1e9:.2f}GB')

In [ ]:
import json
from pathlib import Path
INPUTS = Path(f'{REPO}/kaggle/llm_classify_v2/inputs')
rows = []
for s in ['hard2_test','hard3_test']:
    for line in open(INPUTS/f'{s}.jsonl'):
        r = json.loads(line); r['_split']=s; rows.append(r)
print(f'rows: {len(rows)}')

In [ ]:
import time, json, re
from pathlib import Path

SYSTEM = ('You are an expert algebraist. In magma theory (set G with binary operation \u25c7), '
          'you decide whether a universally-quantified hypothesis h implies a universally-quantified '
          'goal g for ALL magmas. Reason briefly, then output exactly one line: '
          'ANSWER: TRUE or ANSWER: FALSE.')

def to_diamond(s): return s.replace('*','\u25c7')

@torch.inference_mode()
def classify(h, g):
    user = (f'h: forall x y z w u in G, {to_diamond(h)}\n'
            f'g: forall x y z w u in G, {to_diamond(g)}\n\n'
            f'Does h imply g for ALL magmas?')
    msgs = [{'role':'system','content':SYSTEM}, {'role':'user','content':user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False,
        pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
    txt = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    last = txt.strip().split('\n')[-1].upper()
    if 'ANSWER: TRUE' in txt.upper() or last.strip() == 'TRUE': return 'true', txt
    if 'ANSWER: FALSE' in txt.upper() or last.strip() == 'FALSE': return 'false', txt
    if 'TRUE' in last: return 'true', txt
    if 'FALSE' in last: return 'false', txt
    return 'unknown', txt

OUT = Path('/kaggle/working/llm_preds_v2.jsonl')
t0 = time.time(); correct = 0
stats = {'tp_t':0,'fp_t':0,'tp_f':0,'fp_f':0,'unk':0}
with OUT.open('w') as f:
    for i, r in enumerate(rows):
        pred, raw = classify(r['hypothesis'], r['goal'])
        label = r['label']
        if pred == 'true':
            stats['tp_t' if label=='true' else 'fp_t'] += 1
        elif pred == 'false':
            stats['tp_f' if label=='false' else 'fp_f'] += 1
        else: stats['unk'] += 1
        if pred == label: correct += 1
        f.write(json.dumps({'id':r['id'],'split':r['_split'],'label':label,'pred':pred,'raw':raw[-300:]})+'\n')
        f.flush()
        if (i+1) % 30 == 0:
            print(f'[{i+1}/{len(rows)}] correct={correct} stats={stats} t={time.time()-t0:.0f}s', flush=True)
print(f'=== FINAL ===', flush=True)
print(f'accuracy={correct}/{len(rows)} = {correct/len(rows)*100:.1f}%', flush=True)
print(f'stats={stats} time={time.time()-t0:.0f}s', flush=True)